In [20]:
import os
import random
from collections import defaultdict
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
import pandas as pd
import numpy as np
from typing import Dict, List, Tuple


In [ ]:

# Utility functions

def load_vocab(path_entities_txt: str, path_relations_txt: str):
    ent2id, id2ent = {}, {}
    with open(path_entities_txt, 'r') as f:
        for line in f:
            idx, ent = line.strip().split("\t")
            ent2id[ent] = int(idx)
            id2ent[int(idx)] = ent
    rel2id, id2rel = {}, {}
    with open(path_relations_txt, 'r') as f:
        for line in f:
            idx, rel = line.strip().split("\t")
            rel2id[rel] = int(idx)
            id2rel[int(idx)] = rel
    return ent2id, id2ent, rel2id, id2rel

def load_triples_csv(path_csv: str):
    return pd.read_csv(path_csv)

def df_to_triple_list(df: pd.DataFrame, ent2id: Dict[str,int], rel2id: Dict[str,int]):
    triples = []
    for _, row in df.iterrows():
        h, r, t = row['head'], row['relation'], row['tail']
        if h in ent2id and t in ent2id and r in rel2id:
            triples.append((ent2id[h], rel2id[r], ent2id[t]))
    return triples

def build_edge_list_from_df(df: pd.DataFrame, ent2id: Dict[str,int], rel2id: Dict[str,int]):
    src, dst, rels = [], [], []
    for _, r in df.iterrows():
        h, rel, t = r['head'], r['relation'], r['tail']
        if h in ent2id and t in ent2id and rel in rel2id:
            src.append(ent2id[h])
            dst.append(ent2id[t])
            rels.append(rel2id[rel])
    edge_index = torch.tensor([src, dst], dtype=torch.long)
    edge_type  = torch.tensor(rels, dtype=torch.long)
    return edge_index, edge_type

def compute_2hop_drug_gene_disease_support(train_df: pd.DataFrame, ent2id: Dict[str,int]):
    drug_to_genes, gene_to_diseases = defaultdict(set), defaultdict(set)
    for _, row in train_df.iterrows():
        h, rel, t = row['head'], row['relation'], row['tail']
        rel_l = rel.lower()
        if ('target' in rel_l or 'bind' in rel_l or 'interact' in rel_l) and ('drug' in h.lower() or 'drugbank' in h.lower() or 'chem' in h.lower()):
            drug_to_genes[ent2id[h]].add(ent2id[t])
        if ('disease' in t.lower() or 'doid' in t.lower() or 'associate' in rel_l):
            gene_to_diseases[ent2id[h]].add(ent2id[t])
    pair_counts = {}
    for d_id, genes in drug_to_genes.items():
        for g in genes:
            for dis in gene_to_diseases.get(g, []):
                pair_counts[(d_id, dis)] = pair_counts.get((d_id, dis), 0) + 1
    if len(pair_counts) == 0:
        return {}, {}
    max_count = max(pair_counts.values())
    pair_support = {k: v / max_count for k, v in pair_counts.items()}
    return pair_counts, pair_support

class KGDataset(torch.utils.data.Dataset):
    def __init__(self, triples: List[Tuple[int,int,int]]):
        self.samples = triples
    def __len__(self): return len(self.samples)
    def __getitem__(self, idx): return self.samples[idx]

def negative_sample_for_batch(batch_pos: List[Tuple[int,int,int]], num_entities: int, neg_per_pos: int = 5):
    negs = []
    for (h,r,t) in batch_pos:
        for _ in range(neg_per_pos):
            if random.random() < 0.5:
                h_neg = random.randrange(num_entities)
                negs.append((h_neg, r, t))
            else:
                t_neg = random.randrange(num_entities)
                negs.append((h, r, t_neg))
    return negs

def collate_for_loader(batch_pos, num_entities, neg_per_pos=5):
    pos = batch_pos
    neg = negative_sample_for_batch(pos, num_entities, neg_per_pos)
    all_triples = pos + neg
    heads = torch.tensor([h for h,_,_ in all_triples], dtype=torch.long)
    rels  = torch.tensor([r for _,r,_ in all_triples], dtype=torch.long)
    tails = torch.tensor([t for _,_,t in all_triples], dtype=torch.long)
    labels = torch.cat([torch.ones(len(pos)), torch.zeros(len(neg))], dim=0)
    return heads, rels, tails, labels

class ProposedRGCNModel(nn.Module):
    def __init__(self, num_entities, num_relations, dim=64, dropout=0.3):
        super().__init__()
        self.num_entities = num_entities
        self.emb = nn.Embedding(num_entities, dim)
        self.rel_emb = nn.Embedding(num_relations, dim)
        self.fc = nn.Linear(dim*2, 1)
        self.dropout = nn.Dropout(dropout)

    def forward(self, heads, rels, tails, edge_index=None, edge_type=None, pair_support=None):
        """
        heads, rels, tails: tensors of shape [batch_size]
        pair_support: dict {(head_id, tail_id): support_value}
        """
        h_e = self.emb(heads)
        r_e = self.rel_emb(rels)
        t_e = self.emb(tails)

        # Basic 1-hop embedding
        x = h_e + r_e

        # Apply pair_support weighting if provided
        if pair_support is not None:
            # For CPU: iterate batch and get support scalar from dict
            support_weights = []
            for h, t in zip(heads.tolist(), tails.tolist()):
                w = pair_support.get((h, t), 0.0)  # default 0 if not in dict
                support_weights.append(w)
            support_weights = torch.tensor(support_weights, dtype=torch.float32).unsqueeze(-1)  # [batch, 1]
            x = x * (1.0 + support_weights)  # weight 1-hop embedding by 1+support

        # Concatenate with tail embeddings
        x = torch.cat([x, t_e], dim=-1)
        x = self.dropout(x)
        logits = self.fc(x).squeeze(-1)
        return logits


# Training

def train_proposed_rgcn(train_triples, val_triples, num_entities, num_relations,
                        edge_index, edge_type, pair_support=None,
                        epochs=5, batch_size=128, hidden_dim=64, lr=1e-3):
    device = torch.device("cpu")
    dataset = KGDataset(train_triples)
    collate = lambda b: collate_for_loader(b, num_entities, neg_per_pos=3)
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=True, collate_fn=collate)

    model = ProposedRGCNModel(num_entities, num_relations, dim=hidden_dim).to(device)
    optimizer = optim.AdamW(model.parameters(), lr=lr)
    loss_fn = nn.BCEWithLogitsLoss()

    for epoch in range(1, epochs+1):
        model.train()
        total_loss = 0
        for heads, rels, tails, labels in loader:
            heads, rels, tails, labels = heads.to(device), rels.to(device), tails.to(device), labels.to(device)
            optimizer.zero_grad()
            logits = model(heads, rels, tails, edge_index=edge_index, edge_type=edge_type, pair_support=pair_support)
            loss = loss_fn(logits, labels)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
        print(f"Epoch {epoch}/{epochs} - Loss: {total_loss/len(loader):.4f}")

    return model


In [ ]:

if __name__ == "__main__":
    data_dir = r"C:/Users/Manasa/OneDrive/Desktop/Drug_Repurposing_Gnn/Drug_Repurposing_Gnn/data/processed_graph"
    data_dir2 = r"C:/Users/Manasa/OneDrive/Desktop/Drug_Repurposing_Gnn/Drug_Repurposing_Gnn/data"

    device = torch.device("cpu")  # force CPU

    ent2id, id2ent, rel2id, id2rel = load_vocab(
        os.path.join(data_dir, "entities.txt"),
        os.path.join(data_dir, "relations.txt")
    )

    train_df = load_triples_csv(os.path.join(data_dir2, "train_inductive.csv"))
    val_df = load_triples_csv(os.path.join(data_dir2, "val_inductive.csv"))
    test_df = load_triples_csv(os.path.join(data_dir2, "test_inductive.csv"))

    train_triples = df_to_triple_list(train_df, ent2id, rel2id)
    val_triples   = df_to_triple_list(val_df, ent2id, rel2id)
    test_triples  = df_to_triple_list(test_df, ent2id, rel2id)

    edge_index, edge_type = build_edge_list_from_df(train_df, ent2id, rel2id)
    pair_counts, pair_support = compute_2hop_drug_gene_disease_support(train_df, ent2id)

    # Train only Proposed RGCN on CPU
    model = train_proposed_rgcn(
        train_triples=train_triples,
        val_triples=val_triples,
        num_entities=len(ent2id),
        num_relations=len(rel2id),
        edge_index=edge_index,
        edge_type=edge_type,
        pair_support=pair_support,  # keep dict, no dense tensor
        epochs=5,
        batch_size=128,
        hidden_dim=64,
        lr=1e-3
    )

    print("\n=== Training complete ===")


Epoch 1/5 - Loss: 0.3668
Epoch 2/5 - Loss: 0.3196
Epoch 3/5 - Loss: 0.3104
Epoch 4/5 - Loss: 0.3082
Epoch 5/5 - Loss: 0.3075

=== Training complete ===


In [ ]:
torch.save(model.state_dict(), "checkpoints/proposed_rgcn.pt")
print("[Saved] Model weights → checkpoints/proposed_rgcn.pt")


[Saved] Model weights → checkpoints/proposed_rgcn.pt
